In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if(torch.cuda.is_available()):
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'sample_data']


In [6]:
#upload dataset
from google.colab import files
uploaded=files.upload()

Saving Brain_tumar_dataset.zip to Brain_tumar_dataset.zip


In [7]:
#extract the database
import zipfile
import os

zip_path="/content/Brain_tumar_dataset.zip"
extract_path="./content/brain_tumor_data"

with zipfile.ZipFile(zip_path,"r") as zip_ref:
  zip_ref.extractall(extract_path)

print("Dataset Extracted")
print(os.listdir(extract_path))

Dataset Extracted
['Testing', 'Training']


In [13]:
#get folder strucutre
import os

for root,dir,files in os.walk("./content/brain_tumor_data",""):
  level=root.replace("./content/brain_tumor_data","").count(os.sep)

  if level <=2:
    print(" " * level + os.path.basename(root)+"/")

  glioma/
  meningioma/
  pituitary/
  notumor/
 Testing/
  glioma/
  meningioma/
  pituitary/
  notumor/
 Training/
brain_tumor_data/


In [14]:
#create data loaders
import torch
from torch.utils.data import DataLoader,random_split
from torchvision import datasets,transforms

#Reproducibility
SEED=42
torch.manual_seed(SEED)

#setting
IMAGE_SIZE=224
BATCH_SIZE=32

#path
train_dir="./content/brain_tumor_data/Training"
test_dir="./content/brain_tumor_data/Testing"

#imageNet normalization
mean=[0.485,0.456,0.406]
std=[0.229,0.224,0.225]

#training transformations
train_transform=transforms.Compose([
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean,std)
])


#validation/Test transformation
eval_transform=transforms.Compose([
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.transforms.Normalize(mean,std)
])

#load complete training data
full_dataset=datasets.ImageFolder(
    train_dir,
    transform=train_transform
)

#load test dataset
test_dataset=datasets.ImageFolder(
    test_dir,
    transform=eval_transform
)

#data split
train_size=int(0.8*len(full_dataset))
val_size=len(full_dataset)-train_size

train_dataset,val_dataset=random_split(
    full_dataset,
    [train_size,val_size],
    generator=torch.Generator().manual_seed(SEED)

)

#dataloader
train_loader=DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader=DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader=DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


print("Classes:", full_dataset.classes)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))


Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Training samples: 4480
Validation samples: 1120
Test samples: 1600


In [15]:
#create EfficientNet-B0 model
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights


#device select
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device",device)

#load pretrained EfficientNet-B0
weights=EfficientNet_B0_Weights.DEFAULT

model=models.efficientnet_b0(weights=weights)

#freeze the backbone initially
for param in model.features.parameters():
  param.requires_grad= False

#replacr the original classifier with 4 class classifier
in_features=model.classifier[1].in_features

model.classifier=nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features,4)
)

#move model to GPU
model=model.to(device)

#loss function
criterion=nn.CrossEntropyLoss()

#RMSProp optimizer
optimizer=optim.RMSprop(
    model.parameters(),
    lr=0.001,
    alpha=0.0,
    weight_decay=1e-4
)


print("EfficientNet-B0 created successfully!")
print("Number of classes:", 4)
print("Optimizer:", type(optimizer).__name__)


Device cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 103MB/s]


EfficientNet-B0 created successfully!
Number of classes: 4
Optimizer: RMSprop


In [17]:
#train model

import time
import copy
import torch

NUM_EPOCHS=25

best_val_loss=float("inf")
best_model_weight=copy.deepcopy(model.state_dict())

train_losses=[]
val_losses=[]
train_accuracies=[]
val_accuracies=[]

start_time=time.time()


#start train
for epoch in range(NUM_EPOCHS):
  #training
  model.train()

  running_loss=0.0
  correct=0
  total=0

  for images,labels in train_loader:
    images=images.to(device)
    labels=labels.to(device)

    optimizer.zero_grad()

    outputs=model(images)
    loss=criterion(outputs,labels)

    loss.backward()
    optimizer.step()

    running_loss+=loss.item()*images.size(0)
    _,predicted=torch.max(outputs,1)

    total+=labels.size(0)
    correct+=(predicted ++ labels).sum().item()

  epoch_train_loss=running_loss/total
  epoch_train_acc=correct/total


  #validate
  model.eval()

  val_running_loss=0.0
  val_corret=0
  val_total=0

  with torch.no_grad():

    for images,labels in val_loader:

      images=images.to(device)
      labels=labels.to(device)

      outputs=model(images)
      loss=criterion(outputs,labels)

      val_running_loss+=loss.item()*images.size(0)
      _,predicted=torch.max(outputs,1)

      val_total+=labels.size(0)
      val_corret+=(predicted == labels).sum().item()

  epoch_val_loss=val_running_loss/val_total
  epoch_val_acc=val_corret/val_total

  train_losses.append(epoch_train_loss)
  val_losses.append(epoch_val_loss)

  train_accuracies.append(epoch_train_acc)
  val_accuracies.append(epoch_val_acc)


  #save the  mode
  if epoch_val_loss < best_val_loss:
    best_val_loss=epoch_val_loss
    best_model_weight=copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} "
        f"| Train Loss: {epoch_train_loss:.4f} "
        f"| Train Acc: {epoch_train_acc:.4f} "
        f"| Val Loss: {epoch_val_loss:.4f} "
        f"| Val Acc: {epoch_val_acc:.4f} "
        f"| Best"
        )
  else:
    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} "
        f"| Train Loss: {epoch_train_loss:.4f} "
        f"| Train Acc: {epoch_train_acc:.4f} "
        f"| Val Loss: {epoch_val_loss:.4f} "
        f"| Val Acc: {epoch_val_acc:.4f}"
    )

training_time= time.time() +start_time

#restore best model
model.load_state_dict(best_model_weight)

print("\nTraining completed!")
print(f"Training time: {training_time / 60:.2f} minutes")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best validation accuracy: {max(val_accuracies):.4f}")


Epoch 1/25 | Train Loss: 0.3190 | Train Acc: 3.0295 | Val Loss: 0.2577 | Val Acc: 0.9196 | Best
Epoch 2/25 | Train Loss: 0.3364 | Train Acc: 3.0397 | Val Loss: 0.2369 | Val Acc: 0.9170 | Best
Epoch 3/25 | Train Loss: 0.3350 | Train Acc: 3.0283 | Val Loss: 0.2400 | Val Acc: 0.9161
Epoch 4/25 | Train Loss: 0.3326 | Train Acc: 3.0312 | Val Loss: 0.2832 | Val Acc: 0.9036
Epoch 5/25 | Train Loss: 0.3490 | Train Acc: 3.0230 | Val Loss: 0.2641 | Val Acc: 0.9125
Epoch 6/25 | Train Loss: 0.3365 | Train Acc: 3.0413 | Val Loss: 0.2336 | Val Acc: 0.9205 | Best
Epoch 7/25 | Train Loss: 0.3168 | Train Acc: 3.0400 | Val Loss: 0.2687 | Val Acc: 0.9054
Epoch 8/25 | Train Loss: 0.3192 | Train Acc: 3.0306 | Val Loss: 0.2546 | Val Acc: 0.9054
Epoch 9/25 | Train Loss: 0.3385 | Train Acc: 3.0217 | Val Loss: 0.2655 | Val Acc: 0.9107
Epoch 10/25 | Train Loss: 0.3406 | Train Acc: 3.0348 | Val Loss: 0.2617 | Val Acc: 0.9205
Epoch 11/25 | Train Loss: 0.3391 | Train Acc: 3.0317 | Val Loss: 0.2683 | Val Acc: 0.917